In [1]:
#hidden cell to be executed BEFORE the presentation
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import dftpy
from dftpy.ions import Ions
from dftpy.field import DirectField
from dftpy.grid import DirectGrid
from dftpy.functional import LocalPseudo, Functional, TotalFunctional
from dftpy.formats import io
from dftpy.math_utils import ecut2nr
from dftpy.time_data import TimeData
from dftpy.optimization import Optimization
from dftpy.mpi import sprint
from IPython.lib.display import YouTubeVideo
from IPython.display import IFrame
from ase.visualize import view
!wget https://raw.githubusercontent.com/Quantum-MultiScale/DFTpy/refs/heads/dev/examples/DATA/Al_lda.oe01.recpot
PP_list = {'Al': 'Al_lda.oe01.recpot'}
#import fortecubeview

--2026-05-21 11:39:47--  https://raw.githubusercontent.com/Quantum-MultiScale/DFTpy/refs/heads/dev/examples/DATA/Al_lda.oe01.recpot
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 196436 (192K) [text/plain]
Saving to: ‘Al_lda.oe01.recpot’

Al_lda.oe01.recpot  100%[===================>] 191.83K  --.-KB/s    in 0.02s   

2026-05-21 11:39:47 (8.78 MB/s) - ‘Al_lda.oe01.recpot’ saved [196436/196436]



# miniASESMA 2026 -- Accra, Ghana 

# Goals of this lecture + hands-on session
- Basics of the theory behind DFT, Orbital-Free DFT and Kohn-Sham DFT
- KS equations and OF Euler equations
- XC approximations, occupations, k-points
- Basis sets: plane waves (cutoffs)
- Pseudopotentials (local part)
- Sample of OF-DFT and KS-DFT simulations

# The Hohenberg and Kohn theorems
<br>
<br>

$$
\Psi_0 \leftrightarrow n(r) \leftrightarrow v_{eN}(r)
$$

Therefore $n(r)$, $v_{eN}(r)$ or $\Psi_0$ hold the same information. 



In particular:

$$
E \equiv E[\Psi_0] \equiv E[v_{eN}] \equiv E[n]
$$

DFT exploits the latter as follows:

$$
E[n] = T[n] + E_{ee}[n]+E_{eN}[n]+E_{NN}
$$

<br>
<br>
<br>
<center>
<span style="font-size:45pt;"><i>               n(r)</i></span>
</center>
<br>
<br>
<br>
<center>...the density determines everything...</center>

# DFT energy functionals for the KS system

### Single-particle or "Kohn-Sham" DFT

$$
E[n] = T_s[n] + E_{H}[n] + E_{xc}[n] + \int v_{eN}[n](r) n(r) dr  + E_{NN}
$$

where the density $n(r) = \sum_i n_i|\phi_i(r)|^2$. And the single-particle kinetic energy is

$$
T_{s}[n] \equiv T_s[\{\phi_i\}]=  -\frac{1}{2}\sum_i n_i \langle \phi_i | \nabla^2 | \phi_i\rangle = -\frac{1}{2}\sum_i n_i \int \phi_i^*(r) \nabla^2 \phi_i(r) dr
$$

Mind: $T_s \neq T$.



The e-e repulsion and the total kinetic energy are related to $T_s$ and $E_{xc}$ as follows:

$$
E_H[n]=\frac{1}{2}\int \frac{n(r)n(r')}{|r-r'|}drdr'
$$

$$
E_{xc} = \text{Approximated!} \to T[n] + E_{ee}[n] = T_s[n] + E_H[n] + E_{xc}[n]
$$

# DFT energy functionals for the Boson KS (OF) system

### Orbital-free DFT

$$
E[n] = \underbrace{T_{vW}[n] + T_P[n]}_{T_s[n]} + E_{H}[n] + E_{xc}[n] + \int v_{eN}[n](r) n(r) dr  + E_{NN}
$$

where

$$
T_{vW}[n] = -\frac{1}{2}\int \phi^*(r) \nabla^2 \phi(r) dr
$$

where: $\phi(r)=\sqrt{n(r)}$, and 

$$
T_s[n] = \text{Approximated!} \to T_P[n] = T_{s}[n] - T_{vW}[n]
$$

# Solving for the electronic structure

### KS equations for KS and OF-DFT

Define an appropriate Lagrangian:
$$
\mathcal{L}_{KS}[\{\phi_i\}] = E[\{\phi_i\}] - \sum_{ij} \varepsilon_{ij}\left(\langle \phi_j|\phi_i \rangle - \delta_{ij}\right), \qquad \text{for KS-DFT}
$$

Define an appropriate Lagrangian:
$$
\mathcal{L}_{OF}[n] = E[n] -  \mu \left( \int n(r)dr - N\right), \qquad \text{for OF-DFT}
$$

### Let's minimize the Lagrangians to find the ground state KS orbitals and density

For KS-DFT, imposing $\frac{\delta \mathcal{L}_{KS}[\{\phi_i\}]}{\delta \langle \phi_j|}=0$ or just $\frac{\delta \mathcal{L}_{KS}[\{\phi_i\}]}{\delta \phi_j^*(r)}=0$, and choosing the so-called <i>canonical</i> orbitals (i.e., $\varepsilon_{ij}=\varepsilon_{i}\delta_{ij}$),

For OF-DFT, imposing $\frac{\delta \mathcal{L}_{OF}[n]}{\delta \langle \phi|}=0$ or just $\frac{\delta \mathcal{L}_{OF}[n]}{\delta \phi^*(r)}=0$, where $\phi(r) = \sqrt{n(r)}$,

we reach the so-called <span style="color: red;">Kohn-Sham equations</span>:

$$
-\frac{1}{2}\nabla^2 \phi_i(r) + v_s[n](r)\phi_i(r) = \varepsilon_i\phi_i(r), \qquad \text{for KS-DFT}
$$

$$
-\frac{1}{2}\nabla^2 \phi(r) + v_B[n](r)\phi(r) = \mu\phi(r), \qquad \text{for OF-DFT}
$$


# Challenge 2
[Link to the poll](https://rutgers.ca1.qualtrics.com/jfe/form/SV_6LIHtaXrwnoe6LY)

Considering the chain rule of functional differentiation (analogous to regular differentiation where $n(r)$ is treated as the variable and $\phi^*$ and $\phi$ are independent variables):

$$
\frac{\delta F[n]}{\delta \phi_j^*(r)} = \int \frac{\delta F[n]}{\delta n(r')}\frac{\delta n(r')}{\delta \phi_j^*(r)}dr' = \frac{\delta F[n]}{\delta n(r)} \phi_j(r).
$$


1) Derive the KS equations and show that the KS potential is given by:

$$
v_s[n](r) = \frac{\delta E_{H}[n]}{\delta n(r)} + \frac{\delta E_{xc}[n]}{\delta n(r)} + v_{eN}(r)
$$


2) <span style="color: red;">OPTIONAL</span>: Show that the KS potential of OF-DFT is:

$$
v_B[n](r) = \underbrace{\frac{\delta T_{s}[n]}{\delta n(r)} - \frac{\delta T_{vW}[n]}{\delta n(r)}}_{\frac{\delta T_P[n]}{\delta n(r)}} + \frac{\delta E_{H}[n]}{\delta n(r)} + \frac{\delta E_{xc}[n]}{\delta n(r)} + v_{eN}(r)
$$
reaching the KS equation: $-\frac{1}{2}\nabla^2 \sqrt{n(r)} + v_B[n](r)\sqrt{n(r)} = \mu\sqrt{n(r)}$. Where $\mu$ has the same meaning of $\varepsilon_i$ for the KS case.

# Solvers

| | **OF-DFT** | **KS-DFT** |
|:---|:---|:---|
| **Direct minimization** | $$\displaystyle n_0(\mathbf{r}) = \operatorname*{arg\,min}_{n} \left\{ \mathcal{L}_{\mathrm{OF}}[n] \right\}$$ | $$\displaystyle \{\phi_i^0\} = \operatorname*{arg\,min}_{\{\phi_i\}} \left\{ \mathcal{L}_{\mathrm{KS}}[\{\phi_i\}] \right\}$$ |
| **SCF** | $$\displaystyle -\frac{1}{2}\nabla^2 \sqrt{n(\mathbf{r})} + v_B(\mathbf{r})\sqrt{n(\mathbf{r})} = \mu \sqrt{n(\mathbf{r})}$$ | $$\displaystyle -\frac{1}{2}\nabla^2 \phi_i(\mathbf{r}) + v_s(\mathbf{r})\phi_i(\mathbf{r}) = \varepsilon_i \phi_i(\mathbf{r})$$ |

 # Basis sets (discretization)

To be able to run simulations, we need to discretize the space in which the KS and OF wavefunctions / densities live. We will consider plane waves (PW) and Gaussian-type orbitals (GTOs). The general idea is the following:
$$
\phi_i(r) = \sum_\mu^M c_\mu \chi_\mu(r) \text{, where } \chi_\mu \text{ are basis functions}
$$

| | **GTOs** | **PW** |
|:---|:---|:---|
| **Definition** | $$\displaystyle \chi_\mu(\mathbf{r}) = x^{i_\mu} y^{j_\mu} z^{k_\mu}\, \exp\!\left(-\frac{\lVert \mathbf{r}-\mathbf{R}_\mu \rVert^{2}}{2\sigma_\mu^{2}}\right)$$ | $$\displaystyle \chi_\mu(\mathbf{r}) = \frac{1}{\sqrt{\Omega}}\, \mathrm{e}^{\mathrm{i}\, \mathbf{G}_\mu \cdot \mathbf{r}},\quad \mathbf{G}_\mu = \left(\frac{2\pi i_\mu}{a},\,\frac{2\pi j_\mu}{b},\,\frac{2\pi k_\mu}{c}\right)$$ |
| **Location** | $$\mathbf{R}_\mu \in \{\text{centers of atoms}\}$$ | Spread out throughout the simulation cell |
| **Number of functions** | $$\displaystyle \simeq 10 \ \text{per atom}$$ | $$\displaystyle \propto \Omega = a \cdot b \cdot c\text{; generally } 10^{4}\text{ to } 10^{6}$$ |
| **Handling eN potential** | <img src="../figures/science/fullpot.png" alt="GTOs: full potential" width="400"> | <img src="../figures/science/pseudopot.png" alt="PW: pseudopotential" width="400"> |


# Pseudo potentials cannot incorporate core electrons

<table border="1" style="width:100%; text-align:center;">
    <tr>
        <th><center><img src="../figures/science/Electron-Configuration.jpg" alt="econf" width=400 ></center></th>
        <th><center><img src="../figures/science/pseudo_core.png" alt="econf" width=400 ></center></th>
    </tr>
</table>

# Challenge 3

1) Download a pseudopotential from your favorite library
2) Determine the electronic configuration of the ion

In [2]:
additional_files = {'C.pbe-n-rrkjus_psl.1.0.0.UPF' : 'https://pseudopotentials.quantum-espresso.org/upf_files/C.pbe-n-rrkjus_psl.1.0.0.UPF'}
from dftpy.formats import download_files
download_files(additional_files)
!head -n 30 C.pbe-n-rrkjus_psl.1.0.0.UPF


<UPF version="2.0.1">
  <PP_INFO>
Generated using "atomic" code by A. Dal Corso  v.6.3MaX
Author: ADC
Generation date:  4Sep2018
Pseudopotential type: USPP
Element:  C
Functional: PBE
    Suggested minimum cutoff for wavefunctions:  40. Ry
    Suggested minimum cutoff for charge density: 326. Ry
    The Pseudo was generated with a Scalar-Relativistic Calculation
    Local Potential by smoothing AE potential with Bessel fncs, cutoff radius:   1.0000
    Pseudopotential contains additional information for GIPAW reconstruction.
    Valence configuration:
    nl pn  l   occ       Rcut    Rcut US       E pseu
    2S  1  0  2.00      1.000      1.200    -1.010678
    2P  2  1  2.00      0.900      1.400    -0.388489
    Generation configuration:
    2S  1  0  2.00      1.000      1.200    -1.010669
    2S  1  0  0.00      1.000      1.200     3.000000
    2P  2  1  2.00      0.900      1.400    -0.388487
    2P  2  1  0.00      0.900      1.400     0.050000
    Pseudization used: troullier-m

## DFT functionals
### Jacob's ladder of **$E_{\mathrm{xc}}$** (simple picture)

In **Kohn-Sham** DFT, $T_s$ is treated **exactly** (via orbitals), but **$E_{\mathrm{xc}}[n]$** must be modeled. "Going up the ladder" means using **more information** than $n(\mathbf{r})$ alone—usually **better accuracy** and **higher cost**.

| Step (rung) | Family | What the functional can see | Examples (names you may meet) |
|:---|:---|:---|:---|
| **1** | **LDA** | local density $n(\mathbf{r})$ only | SVWN, PW92, PZ, … |
| **2** | **GGA** | $n$ **and** gradient $\nabla n$ | PBE, BLYP, revPBE, … |
| **3** | **meta-GGA** | adds **kinetic density** $\tau_s(\mathbf{r}) = \frac{1}{2}\sum_i f_i \lvert \nabla\phi_i \rvert^2$ | SCAN, $\mathrm{r}^2\mathrm{SCAN}$, TPSS, … |
| **4** | **Hybrids** | mixes in **exact exchange** (Fock operator) with DFT exchange | PBE0, B3LYP, HSE06, … |
| **5+** | **Beyond** | e.g. **double hybrids**, RPA / many-body perturbation ideas | more expensive "high rungs" |

**Hubbard functionals** which are inspired by GW.
- GGA+U, GGA+U+V

**Takeaway:** pick the **lowest rung** that is accurate enough for your problem and budget; **LDA/GGA** remain workhorses in solid state; **meta-GGA / hybrids** are common for chemistry and band gaps.

## KEDF Approximations

### Kinetic energy (KE) in orbital-free DFT

In **OF-DFT**, the **non-interacting kinetic energy** $T_s[n]$ is not obtained from orbitals; it is a functional of the density:
$$
T_s[n] \approx T_{\mathrm{vW}}[n] + T_{\mathrm{P}}[n]
$$

### $T_s[n]$: (semi)local and nonlocal functionals
<br>
<center>
    <img src="../figures/science/local_nonlocal.png" width=1600 />
<p>Wenhui Mi, MP JCP (2018) • Wenhui Mi, MP PRB (2019) • Xuecheng Shao, WM, MP PRB (2021)
Xuecheng Shao, WM, MP JPCL (2021) • Xuecheng Shao, WM, MP JCTC (2021) • Wenhui Mi, MP JPCL (2020)</p>
    </center>


# The Self-Consistent Field Method
The KS equations $-\frac{1}{2}\nabla^2 \phi_i(r) + v_s(r)\phi_i(r) = \varepsilon_i\phi_i(r)$ feature $v_s(r)$ which depends on the density:
<br>
<br>
$$
v_s[n](r) = \frac{\delta E_{H}[n]}{\delta n(r)} + \frac{\delta E_{xc}[n]}{\delta n(r)} + v_{eN}(r)
$$

But the density depends on the KS orbitals, $\{\phi_i(r)\}$:

$$
n(r) = \sum_i n_i |\phi_i(r)|^2\,\, n_i \text{ are the occupation numbers.}
$$

# Setting up QEpy and QE's input file for KS-DFT

In [3]:
from qepy.driver import Driver
from qepy.io import QEInput

In [4]:
IFrame(src="https://www.quantum-espresso.org/Doc/INPUT_PW.html",width=1250, height=600)

### QEpy's input file is a dictionary containing QE's input keywords

In [5]:
from dftpy.formats import download_files
additional_files = {'Al.pbe-nl-kjpaw_psl.1.0.0.UPF' : 'https://pseudopotentials.quantum-espresso.org/upf_files/Al.pbe-nl-kjpaw_psl.1.0.0.UPF'}
download_files(additional_files)

In [6]:
qe_options = {
    '&control': {
        'calculation': "'scf'",
        'pseudo_dir': "'./'",
    },
    '&system': {
        'ibrav' : 0,
        'degauss': 0.005,
        'ecutwfc': 30,
        'occupations': "'smearing'"
    },
    'atomic_species': ['Al  26.98 Al.pbe-nl-kjpaw_psl.1.0.0.UPF'],
    'k_points gamma': [],
}

In [7]:
options = {
    '&electrons': {
        'mixing_beta': 0.5},
    'cell_parameters angstrom':[
        '0.     2.025  2.025',
        '2.025  0.     2.025',
        '2.025  2.025  0.   '],
    'atomic_positions crystal': ['Al    0.0  0.0  0.0'],
    'k_points automatic': ['6 6 6 1 1 1'],
}

# Let's look into
## $\bullet$ k-points?

In [8]:
qe_options = QEInput.update_options(options, qe_options=qe_options)
driver = Driver(qe_options=qe_options, logfile=True)
%timeit -n1 -r1 driver.scf() 

2.04 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [9]:
driver=Driver(qe_options=qe_options, iterative = True, logfile='tmp.out') 
for i in range(60):
    driver.diagonalize()
    driver.mix()
    converged = driver.check_convergence()
    print ('Iter: ',i,' - Conv: ', driver.get_scf_error())
    if converged : break
driver.calc_energy()

Iter:  0  - Conv:  0.001437642087473308
Iter:  1  - Conv:  6.856965727256235e-05
Iter:  2  - Conv:  5.649531351467042e-07


-39.50195892190139

# Bloch theorem and k-point sampling

1) Solids have periodic potentials:
$$
v_{eN}(r+n\hat a) = v_{eN}(r),\,\, \text{where } \hat a \text{ is a lattice vector.}
$$

<center><img src="../figures/science/periodic_pot.png" width=1300 /></center>

2) Bloch theorem states that the group of translations comes with 3 quantum numbers $\vec k = (k_a, k_b, k_c)$ representing translations along the 3 lattice vectors. The wavefunctions will be labelled by $k$ (dropped $\vec k$ for a lighter notation):
$$
\phi_i \to \phi_{ik} = e^{ik\cdot r} u_{ik}(r), \,\, k\in \text{First Brillouin Zone (FBZ)}
$$

FBZ: the set of $k$ vectors generating a phase $e^{ik\cdot r}$ that is periodic by no less than a full cell length ($k_{a/b/c} \leq 0.5$).

# K-point sampling example with QEpy

1) First, let's see how many orbitals (bands) we have.

In [10]:
print(f"Number of orbitals (bands) considered by QE: {driver.get_number_of_bands()}") 

Number of orbitals (bands) considered by QE: 6


2) Let's take a look at the irreducible k-points considered by QE.

In [11]:
driver.get_ibz_k_points()

array([[ 0.08333333,  0.08333333,  0.08333333],
       [ 0.08333333,  0.08333333,  0.25      ],
       [ 0.08333333,  0.08333333,  0.41666667],
       [ 0.08333333,  0.08333333, -0.41666667],
       [ 0.08333333,  0.08333333, -0.25      ],
       [ 0.08333333,  0.08333333, -0.08333333],
       [ 0.08333333,  0.25      ,  0.25      ],
       [ 0.08333333,  0.25      ,  0.41666667],
       [ 0.08333333,  0.25      , -0.41666667],
       [ 0.08333333,  0.25      , -0.25      ],
       [ 0.08333333,  0.25      , -0.08333333],
       [ 0.08333333,  0.41666667,  0.41666667],
       [ 0.08333333,  0.41666667, -0.41666667],
       [ 0.08333333,  0.41666667, -0.25      ],
       [ 0.08333333,  0.41666667, -0.08333333],
       [ 0.08333333, -0.41666667, -0.41666667],
       [ 0.08333333, -0.41666667, -0.25      ],
       [ 0.08333333, -0.25      , -0.25      ],
       [ 0.25      ,  0.25      ,  0.25      ],
       [ 0.25      ,  0.25      ,  0.41666667],
       [ 0.25      ,  0.25      , -0.416

3) Check that the total number of electrons is the sum of all occupations (times the k-point weight) over all k-points and all bands:

In [12]:
N=[]
for i in range(driver.get_number_of_k_points()):
    N.append(driver.get_occupation_numbers(kpt=i).sum())
N=np.asarray(N)
N.sum()

2.999999999940215

# run OF-DFT simulations?

In [13]:
from ase.build import bulk
atoms = bulk('Al', 'fcc', a=4.05, cubic=True)
ions = Ions.from_ase(atoms)
view(ions)
nr = ecut2nr(ecut=35, lattice=ions.cell)
grid = DirectGrid(lattice=ions.cell, nr=nr)

In [14]:
nr = ecut2nr(ecut=35, lattice=ions.cell)
grid = DirectGrid(lattice=ions.cell, nr=nr)
PSEUDO = LocalPseudo(grid = grid, ions=ions, PP_list=PP_list)
rho_ini = DirectField(grid=grid)
rho_ini[:] = ions.get_ncharges()/ions.cell.volume
HARTREE = Functional(type='HARTREE')
XC = Functional(type='XC',name='LDA')
KE = Functional(type='KEDF', name='TFvW')

setting key: Al -> Al_lda.oe01.recpot


usage: ase [-h] [--version] [-T]
           {help,info,test,gui,db,run,band-structure,build,dimensionality,eos,ulm,find,nebplot,convert,reciprocal,completion,diff,exec}
           ...
ase: error: ModuleNotFoundError: No module named '_tkinter'
To get a full traceback, use: ase -T gui ...


In [15]:
evaluator = TotalFunctional(KE=KE, XC=XC, HARTREE=HARTREE, PSEUDO=PSEUDO)
optimization_options = {'econv' : 1e-3*ions.nat}
opt = Optimization(EnergyEvaluator=evaluator, optimization_options = optimization_options,
        optimization_method = 'TN')
%timeit -n1 -r2 rho = opt.optimize_rho(guess_rho=rho_ini)

Step    Energy(a.u.)            dE              dP              Nd      Nls     Time(s)         
0       -8.090977710718E+00     -8.090978E+00   7.877088E-01    1       1       2.931368E-01    
1       -8.273130665182E+00     -1.821530E-01   7.745404E-02    2       2       3.883111E-01    
2       -8.280424972418E+00     -7.294307E-03   7.026526E-03    6       2       5.085120E-01    
3       -8.281101144768E+00     -6.761724E-04   5.767574E-04    5       3       5.726249E-01    
4       -8.281133098850E+00     -3.195408E-05   5.322854E-05    4       2       6.068130E-01    
#### Density Optimization Converged ####
Chemical potential (a.u.): 0.3011451142214292
Chemical potential (eV)  : 8.194575952431476
Step    Energy(a.u.)            dE              dP              Nd      Nls     Time(s)         
0       -8.090977710718E+00     -8.090978E+00   7.877088E-01    1       1       1.194096E-02    
1       -8.273130665182E+00     -1.821530E-01   7.745404E-02    2       2       4.025006E-02

# Challenge 4

1) Write you own SCF code using DFTpy
[`scf_challenge.ipynb`](../tutorials/scf_challenge.ipynb)
